<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Prompt Engineering für Agenten
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Der System-Prompt legt fest, wie der Meeting- & Research-Briefing-Agent seine Rolle versteht, wann er Tools einsetzt (**Planen und Handeln**) und wo er hart stoppen muss (**Prüfen** — siehe Hard Limits in Kapitel 6). Prompt Engineering ist damit keine reine Stil-Frage, sondern Teil der Steuerungslogik des Agentensystems.


In [1]:
#@title  🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
# Globales Tracing aus: Nur der Referenzlauf in Kapitel 6 wird erfasst.
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "M03-Prompt-Engineering"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.4.3
langchain-anthropic                      1.7.5
langchain-chroma                         1.1.0
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.6.6
langchain-google-genai                   4.4.0
langchain-ollama                         1.1.0
langchain-openai                         1.6.7
langchain-protocol                       0.0.19
langchain-text-splitters                 1.1.3
langgraph                                1.2.11
langgraph-checkpoint                     4.2.0
langgraph-checkpoint-sqlite              3.1.1
langgraph-prebuilt                       1.1.0
langgraph-sdk                            0.4.4

IP-Adresse: 34.24.74.175
Hostname: 175.74.24.34

# 1 | Übersicht
---


Prompts sind die zentrale Steuerungsschicht zwischen Entwickler, Modell und Tools. Was im Prompt nicht klar geregelt ist, bleibt dem Modell überlassen: Rolle, Tool-Nutzung, Grenzen, Ausgabeformat und Stoppbedingungen.


In [2]:
#@markdown   <p><font size="4" color='green'>  Prompt-Engineering-Workflow</font> </br></p>


# Prompt-Engineering-Workflow
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart TB
    INPUT([Aufgabe]) --> SYSTEM[System-Prompt
Rolle & Kontext]
    INPUT --> FEW[Few-Shot
Beispiele]
    SYSTEM --> LLM[LLM]
    FEW --> LLM
    LLM --> TOOL{Tool
nötig?}
    TOOL -- ja --> CALL[Tool-Aufruf]
    CALL --> LLM
    TOOL -- nein --> OUT([Antwort])
"""
mermaid(diagram, width=700)

Im Kurs wird das am Meeting-Briefing-Agent geübt: derselbe fachliche Kern aus M02 wird durch bessere Prompts zuverlässiger, kontrollierbarer und leichter prüfbar.


In [3]:
# Imports für das Modul
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model(BASELINE)

# 2 | ChatPromptTemplate
---



`ChatPromptTemplate` trennt klar zwischen Rollen (`system`, `user`, `ai`) und Variablen.
Das ist robuster als Prompt-Strings zusammenzukleben.


**`load_prompt()` - Prompts als versionierte Assets**

Statt Prompts direkt im Code zu schreiben, lädt `load_prompt()` ein Markdown-Dokument aus einem lokalen Pfad oder einer GitHub-URL und gibt ein fertiges `ChatPromptTemplate` zurück.

```python
**Aus GitHub (Blob- und Tree-URLs werden automatisch in Raw-Links umgewandelt)**
prompt = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m03_research_template_prompt.md",
    mode="T"   # "T" = ChatPromptTemplate | "S" = plain String
)

**Oder lokal:**
prompt = load_prompt("../05_prompt/m03_research_template_prompt.md", mode="T")
```


**Prompt-Format (Markdown mit YAML-Frontmatter):**
```markdown
---
name: research_template
description: Erklärt Briefing-Agent-Konzepte für verschiedene Zielgruppen
variables: [thema, zielgruppe]
---

## system
Rolle: Meeting-Briefing-Agent für den KI-Agenten-Kurs.

## user
Erkläre {thema} für {zielgruppe} in drei Sätzen.
```



**Vorteile gegenüber Inline-Prompts:**

| | Inline im Code | `load_prompt()` aus Datei |
|---|---|---|
| **Versionierung** | Im Code-Commit versteckt | Eigener Commit, diff-bar |
| **Übersichtlichkeit** | Code und Prompts vermischt | Klar getrennt |
| **Wiederverwendung** | Copy-Paste | Eine Datei, viele Notebooks |
| **Zusammenarbeit** | Erfordert Code-Kenntnisse | Einfaches Markdown |

> Das `05_prompt/`-Verzeichnis im Repository ist damit ein **Prompt Hub** – alle Prompts an einem Ort, versioniert per Git. Alle Notebooks dieses Kurses nutzen dieses Muster.

In [4]:
prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m03_research_template_prompt.md", mode="T")

messages = prompt.invoke({
    "thema": "Retrieval Augmented Generation",
    "zielgruppe": "Einsteiger"
})

for msg in messages.messages:
    print(f"{msg.type.upper()}: {msg.content}")

SYSTEM: Rolle: Meeting- & Research-Briefing-Agent für den KI-Agenten-Kurs.
Erkläre knapp, fachlich sauber und mit Bezug auf Projektunterlagen oder wissenschaftliche Dokumente.
HUMAN: Erkläre Retrieval Augmented Generation für Einsteiger in drei Sätzen.


In [5]:
parser = StrOutputParser()

chain = prompt | llm | parser

response = chain.invoke({"thema": "Korpusabdeckung", "zielgruppe": "Briefing-Agent-Entwicklung"})
mprint(response)

Korpusabdeckung bezeichnet, wie vollständig und repräsentativ die für einen Briefing-Agenten verfügbaren Dokumente die relevanten Themen, Quellen, Zeiträume und Anwendungsfälle abbilden. Eine hohe Korpusabdeckung reduziert das Risiko, dass der Agent wichtige Informationen oder Gegenpositionen übersieht, und verbessert dadurch Recherche, Zusammenfassungen und Briefings. Sie sollte anhand definierter Kriterien wie Quellenvielfalt, Aktualität, thematischer Vollständigkeit und Dokumentqualität überprüft werden.

# 3 | System Prompts für Tool-Calling
---



Für Agenten mit Tools sollte der `system`-Prompt explizit festlegen:
- wann ein Tool verwendet wird
- wann **kein** Tool verwendet wird
- welches Ausgabeformat erwartet wird


In [6]:
from langchain.agents import create_agent
from langchain_core.tools import tool

In [7]:
@tool
def research_signal(text: str) -> str:
    """Extrahiert einfache Briefing-Signale aus einer Anfrage."""
    signale = [wort for wort in ["rag", "retrieval", "evaluation", "agent", "embedding"] if wort in text.lower()]
    return ", ".join(signale) if signale else "Keine klaren Briefing-Signale gefunden."

@tool
def korpus_check(thema: str) -> str:
    """Prüft grob, ob ein Thema zu den Kurskorpus-Schwerpunkten passt."""
    kursnah = ["rag", "retrieval", "evaluation", "agent", "embedding"]
    if any(wort in thema.lower() for wort in kursnah):
        return "Thema passt zum Wissenskorpus."
    return "Keine sichere Korpusabdeckung erkennbar."

In [8]:
system_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m03_research_system_prompt.md", mode="S")

agent = create_agent(
    model=WORKER,
    tools=[research_signal, korpus_check],
    system_prompt=system_prompt,
)

result = agent.invoke({"messages": [HumanMessage(content="Prüfe RAG und Evaluation als Thema für den Wissenskorpus.")]})
mprint(result["messages"][-1].content)

Ja, **RAG und Evaluation** passt zum Wissenskorpus.

Wenn du willst, kann ich als Nächstes auch kurz einordnen:
- **welche Teilaspekte** von RAG/Evaluation besonders relevant sind,
- oder **eine mögliche Gliederung** für den Korpus dazu skizzieren.

# 4 | Few-Shot Examples
---



Few-Shot-Beispiele helfen, Stil und Struktur der Antworten zu stabilisieren.
Nutze wenige, aber repräsentative Beispiele.


In [9]:
zero_shot = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m03_research_zero_shot_prompt.md", mode="T")
few_shot = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m03_research_few_shot_prompt.md", mode="T")

In [10]:
frage = "Warum verbessert RAG die Zuverlässigkeit eines Meeting-Briefing-Agenten?"
for name, p in [("Zero-Shot", zero_shot), ("Few-Shot", few_shot)]:
    run_cfg = {
        "run_name": f"M03_Kap4_{name.replace('-', '')}",
        "tags": ["M03", "few-shot", "research", name.lower().replace("-", "")]
    }
    out = (p | llm).with_config(**run_cfg).invoke({"frage": frage})
    print()
    print(f"{name}: {out.content.strip()}")


Zero-Shot: anwendung

Few-Shot: anwendung


# 5 | Prompt-Varianten testen
---



Ein einfacher Vergleich hilft, bessere Prompts datenbasiert auszuwählen.
Unten testen wir mehrere System-Prompts auf dieselbe Anfrage.


In [11]:
variants = {
    "knapp": "Antworte in maximal 2 Sätzen.",
    "didaktisch": "Erkläre schrittweise und nenne ein kurzes Beispiel aus dem Meeting-Briefing-Agent.",
    "kritisch": "Zeige Antwort, Unsicherheit und eine mögliche Fehlerquelle.",
}

variant_prompt = ChatPromptTemplate([
    ("system", "Rolle: Meeting-Briefing-Agent für den KI-Agenten-Kurs. {instruction}"),
    ("user", "{q}"),
])

In [12]:
user_question = "Wann sollte ein Meeting-Briefing-Agent ein Retrieval-Tool nutzen?"

for name, instruction in variants.items():
    run_cfg = {
        "run_name": f"M03_Kap5_Variant_{name}",
        "tags": ["M03", "prompt-variants", "research", name]
    }
    answer = (variant_prompt | llm).with_config(**run_cfg).invoke({"instruction": instruction, "q": user_question})
    mprint("### Variante: " + name)
    mprint('---')
    mprint(answer.content.strip())

### Variante: knapp

---

Ein Meeting-Briefing-Agent sollte ein Retrieval-Tool nutzen, wenn aktuelle, spezifische oder nicht im Kontext enthaltene Informationen benötigt werden, etwa frühere Protokolle, offene Aufgaben, Teilnehmerdetails oder relevante Dokumente. Nicht nötig ist es bei allgemeinen Fragen oder wenn alle erforderlichen Informationen bereits im Kontext vorliegen.

### Variante: didaktisch

---

Ein Meeting-Briefing-Agent sollte ein **Retrieval-Tool** nutzen, wenn er Informationen aus externen oder internen Wissensquellen abrufen muss.

### Schrittweise Entscheidung

1. **Prüfen, ob die benötigten Informationen bereits im Auftrag enthalten sind**  
   Wenn alle Fakten im Prompt stehen, ist kein Retrieval nötig.

2. **Erkennen, ob Wissen aus Dokumenten benötigt wird**  
   Retrieval ist sinnvoll bei:
   - früheren Meeting-Protokollen
   - CRM-Daten
   - Projektunterlagen
   - E-Mails oder Tickets
   - Richtlinien und Verträgen
   - aktuellen Unternehmensinformationen

3. **Auf Aktualität und Genauigkeit achten**  
   Wenn sich Informationen geändert haben können oder konkrete Belege erforderlich sind, sollte der Agent die Quelle abrufen, statt sich nur auf sein Modellwissen zu verlassen.

4. **Gezielt suchen**  
   Der Agent sollte möglichst präzise Suchanfragen verwenden, etwa nach Projektname, Kundennamen, Datum oder Thema.

5. **Quellen im Briefing kenntlich machen**  
   Die gefundenen Informationen sollten idealerweise mit Dokumenttitel, Datum oder Link belegt werden.

### Kurzes Beispiel

Vor einem Kundentermin soll der Agent ein Briefing erstellen. Er sucht per Retrieval-Tool:

- das letzte Gesprächsprotokoll,
- offene Support-Tickets,
- den aktuellen Projektstatus.

Daraus erstellt er etwa:

> „Im letzten Gespräch vom 12. September wurde eine Integration bis Ende Oktober vereinbart. Aktuell sind noch zwei Support-Tickets offen.“

Ohne Retrieval hätte der Agent möglicherweise veraltete oder unbelegte Informationen verwendet.

### Variante: kritisch

---

Ein Meeting-Briefing-Agent sollte ein **Retrieval-Tool** nutzen, wenn er für das Briefing auf aktuelle, unternehmensinterne oder kontextabhängige Informationen zugreifen muss, die nicht zuverlässig im Modellwissen enthalten sind.

Typische Fälle:

- **Vergangene Meetings:** Protokolle, Entscheidungen, offene Aufgaben und Verantwortliche
- **Aktuelle Projektdaten:** Statusberichte, Zeitpläne, Risiken und Meilensteine
- **Teilnehmerinformationen:** Rollen, Zuständigkeiten und frühere Beiträge
- **Relevante Dokumente:** Verträge, Anforderungen, Präsentationen oder Richtlinien
- **Änderungen seit dem letzten Meeting:** Neue Tickets, E-Mails, Kommentare oder Kennzahlen
- **Quellenbasierte Aussagen:** Wenn Aussagen mit Links, Dokumenten oder Zitaten belegt werden sollen

Ein Retrieval-Tool ist besonders sinnvoll, wenn die Antwort von der **Aktualität**, **Genauigkeit** oder **Nachvollziehbarkeit** der Daten abhängt. Für eine einfache Agenda oder eine allgemeine Gesprächsstruktur ist es dagegen meist nicht nötig.

**Beispiel:**  
Vor einem Projektstatus-Meeting sollte der Agent die letzten Protokolle, den aktuellen Projektstatus und offene Jira-Tickets abrufen, statt sich nur auf den bisherigen Gesprächsverlauf zu stützen.

**Unsicherheit:** Ob Retrieval erforderlich ist, hängt davon ab, wie aktuell und vollständig der bereits vorhandene Kontext ist. Wenn alle relevanten Informationen im Prompt enthalten und vertrauenswürdig sind, kann ein zusätzlicher Abruf entfallen.

**Mögliche Fehlerquelle:** Das Retrieval kann veraltete, unvollständige oder falsch zugeordnete Dokumente liefern. Deshalb sollte der Agent Quellen nach Aktualität und Relevanz filtern und bei widersprüchlichen Informationen auf die Unsicherheit hinweisen.

# 6 | Reasoning-Strategien für Agenten
---

Reasoning im Kurs bedeutet nicht, versteckte Modellgedanken auszugeben. Für kontrollierte Agenten zählen **sichtbare Zwischenartefakte**, die geprüft werden können: Aufgabenzerlegung, Plan, gewählte Tools, Quellenstatus, Qualitätskriterien und Entscheidung am Gate.

| Strategie | Sichtbares Artefakt | Geeignet für | Nicht geeignet für |
|---|---|---|---|
| **ReAct / TAO** | Tool-Aufruf und Observation | Schrittweises Handeln mit Tools | Lange Aufgaben ohne Gesamtplan |
| **Plan-and-Execute** | Kurzer Arbeitsplan + Status je Schritt | Mehrteilige Briefings, Recherche + Synthese | Sehr einfache Einzelfragen |
| **Reflection / Self-Correction** | Prüfergebnis + konkrete Korrektur | Qualitäts-Gates, Quellenprüfung | Als endlose Selbstkritik ohne Abbruchregel |
| **Tree-of-Options** | 2-3 Optionen mit Auswahlkriterium | Routing-Entscheidungen, Architekturwahl | Vollständige Baumexplosion in Kursnotebooks |

**Kursregel:** Keine Aufforderung wie „zeige deine Chain-of-Thought“. Stattdessen: „Gib Plan, Kriterien, Tool-Wahl und Ergebnis knapp als prüfbare Felder aus.“


In [13]:
# Kostenfreier Reasoning-Selfcheck: Strategie anhand der Aufgabe auswählen
reasoning_patterns = {
    "react": {
        "trigger": ["tool", "lookup", "nachschlagen", "berechnen"],
        "artefakt": "Nächster Tool-Aufruf + Observation",
    },
    "plan_and_execute": {
        "trigger": ["mehrere", "briefing", "projekt", "recherche"],
        "artefakt": "Plan mit 3-5 Schritten und Status je Schritt",
    },
    "reflection": {
        "trigger": ["prüfen", "prüfe", "qualität", "quelle", "belegt", "korrektur"],
        "artefakt": "Score, Feedback und konkrete Nachbesserung",
    },
    "tree_of_options": {
        "trigger": ["entscheiden", "entscheide", "routing", "geroutet", "alternative", "option"],
        "artefakt": "Optionen + Auswahlkriterium",
    },
}


def waehle_reasoning_pattern(aufgabe: str) -> dict:
    text = aufgabe.lower()
    scores = {
        name: sum(1 for signal in info["trigger"] if signal in text)
        for name, info in reasoning_patterns.items()
    }
    pattern = max(scores, key=scores.get)
    if scores[pattern] == 0:
        pattern = "react"
    return {
        "pattern": pattern,
        "sichtbares_artefakt": reasoning_patterns[pattern]["artefakt"],
        "scores": scores,
    }


reasoning_tests = [
    "Erstelle ein Briefing mit Recherche, Quellen und kurzer Synthese.",
    "Prüfe, ob die Antwort ausreichend belegt ist.",
    "Entscheide, ob die Anfrage zu RAG oder Security geroutet wird.",
]

for test in reasoning_tests:
    entscheidung = waehle_reasoning_pattern(test)
    print(test)
    print("->", entscheidung["pattern"], "|", entscheidung["sichtbares_artefakt"])

assert waehle_reasoning_pattern(reasoning_tests[0])["pattern"] == "plan_and_execute"
assert waehle_reasoning_pattern(reasoning_tests[1])["pattern"] == "reflection"


Erstelle ein Briefing mit Recherche, Quellen und kurzer Synthese.
-> plan_and_execute | Plan mit 3-5 Schritten und Status je Schritt
Prüfe, ob die Antwort ausreichend belegt ist.
-> reflection | Score, Feedback und konkrete Nachbesserung
Entscheide, ob die Anfrage zu RAG oder Security geroutet wird.
-> tree_of_options | Optionen + Auswahlkriterium


<p><font color='black' size="5">
Prompt-Pattern: Reasoning sichtbar machen, ohne interne Gedankenkette
</font></p>

Für Agenten-Prompts ist diese Formulierung robuster und sicherer:

```text
Arbeite mit sichtbaren Kontrollfeldern:
- plan: maximal 4 Schritte
- tool_choice: welches Tool und warum in einem Satz
- evidence_status: ausreichend | unsicher | nicht_im_korpus
- decision: antworten | nachrecherchieren | eskalieren

Gib keine ausführliche interne Gedankenkette aus.
```

Diese Felder werden später in LangGraph zu State-Feldern, Routing-Kriterien und Evaluationsmetriken.


# 7 | Strukturierte Agenten-Prompts: Production Pattern
---


LangGraph-Produktions-Prompts folgen einem bewährten Muster mit 5 Patterns,
die Agenten zuverlässiger und kontrollierbarer machen.


<p><font color='black' size="5">
Pattern 1: XML-Tags als Struktur-Trenner
</font></p>

LLMs folgen klar abgegrenzten Sektionen zuverlässiger als Fließtext.
Jede Sektion hat einen eigenen kognitiven Zweck.

```python
**Unstrukturiert: Modell muss Regeln aus Fließtext extrahieren**
system = "Rolle: Meeting-Briefing-Agent. Nutze Tools. Sei effizient. Stoppe wenn fertig."

**Strukturiert: klare Trennung von Rolle, Prozess und Grenzen**
system = """
<Task>Was der Agent tun soll</Task>
<Instructions>Wie er vorgehen soll</Instructions>
<Hard Limits>Wann er stoppen soll</Hard Limits>
"""
```

Fehlt die XML-Struktur, übersieht das Modell häufiger Regeln, die weiter unten im Prompt stehen - besonders bei langen System-Prompts mit mehr als 500 Tokens.




<p><font color='black' size="5">
Pattern 2: Explizite Tool-Steuerung
</font></p>

Ohne Anweisung nutzen Agenten Tools opportunistisch. Explizite Reflexions-Aufforderung erzwingt Bewertung nach jedem Schritt.

```python
"KRITISCH: Nach jedem Tool-Ergebnis bewerten: Reicht der Kontext für eine belegbare Antwort?"
```




<p><font color='black' size="5">
Pattern 3: Quantitative Limits statt qualitativer Anweisungen
</font></p>

```python
**Qualitativ: nicht handlungsleitend genug**
"Sparsam mit Tool-Calls umgehen."

**Quantitativ: konkret und messbar**
"Tool-Budget: maximal 3 Retrieval-Aufrufe. Nach 3 Aufrufen immer stoppen."
```



<p><font color='black' size="5">
Pattern 4: Explizite Stop-Bedingungen
</font></p>

Agenten neigen zu Over-Research. Ohne Stop-Kriterien laufen sie bis zum `recursion_limit`.

```python
"""Sofort stoppen wenn:
- Die Frage vollständig mit verfügbarem Kontext beantwortet werden kann
- Die letzte Suche keine neuen Informationen brachte
- 3 relevante Quellen gefunden wurden
"""
```




<p><font color='black' size="5">
Pattern 5: Strategie vorgeben (breit -> eng)
</font></p>

Für Recherche-Agenten ist eine explizite Suchstrategie wichtig:
+ zuerst grob einordnen,
+ dann gezielt vertiefen,
+ danach Antwort synthetisieren.


In [14]:
# Strukturierter System-Prompt mit allen 5 Patterns
structured_system_prompt = """

Rolle: Meeting-Briefing-Agent für den KI-Agenten-Kurs.

<Task>
Beantworte Fachfragen zu RAG, Retrieval, Evaluation und Agenten knapp und nachvollziehbar.
</Task>

<Instructions>
1. Frage einordnen: Begriff, Vergleich, Anwendung oder außerhalb des Korpus?
2. Tool nur nutzen, wenn Briefing-Signale oder Korpusabdeckung geprüft werden müssen.
3. Nach jedem Tool-Ergebnis prüfen: Reicht der Kontext für eine belegbare Antwort?
4. Unsicherheit ausdrücklich nennen, wenn die Korpusabdeckung unklar ist.
</Instructions>

<Hard Limits>
Tool-Budget: maximal 2 Tool-Aufrufe.
Maximale Antwortlänge: 120 Wörter.
Sofort antworten, wenn Begriff und Korpusbezug klar sind.
</Hard Limits>"""

In [15]:
@tool
def beispiel_quellenhinweis(thema: str) -> str:
    """Gibt einen Demo-Hinweis auf mögliche Quellenbindung zurück."""
    return f"Für '{thema}' sollte die Antwort mit Fundstelle, Paper-Titel oder Chunk-ID belegt werden."

In [16]:
agent = create_agent(
    model=WORKER,
    tools=[beispiel_quellenhinweis],
    system_prompt=structured_system_prompt,
)

result = agent.invoke({"messages": [HumanMessage(content="Wie sollte ein Meeting-Briefing-Agent Antworten zu RAG belegen?")]})
mprint(result["messages"][-1].content)

Ein Meeting-Briefing-Agent sollte Antworten zu RAG **kurz, nachvollziehbar und quellengebunden** belegen:

- **Begriff sauber einordnen**: z. B. Definition, Vergleich, Anwendung.
- **Antwort an Korpus koppeln**: nur Aussagen machen, die durch das verfügbare Briefing-/Korpusmaterial gedeckt sind.
- **Belege explizit nennen**: z. B. Dokumenttitel, Abschnitt, Seite oder Passage.
- **Unsicherheit markieren**, wenn die Korpusabdeckung unklar ist.
- **Keine Spekulation**: bei fehlender Abdeckung sagen, dass die Antwort außerhalb des Korpus liegt.
- **Kurz zitieren oder paraphrasieren**: bevorzugt mit direktem Bezug auf die Quelle.

Wenn du willst, kann ich dir auch ein **Musterformat für RAG-belegte Antworten** geben.

# 8 | LangSmith: Prompt-Inspektion
---



LangSmith macht Prompt-Experimente nachvollziehbar: Welcher Prompt erzeugte welche Antwort?

**Empfehlung im Kursbetrieb:**
- `run_name` pro Experiment setzen
- `tags` für Modul/Kapitel verwenden
- bei Vergleichen immer denselben Input nutzen
- nur den Referenzlauf explizit mit LangSmith tracen, damit keine unnötigen Traces entstehen

**Hinweis (EU):** Endpoint `https://eu.api.smith.langchain.com` – bereits in der Setup-Cell konfiguriert.

In [17]:
# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M03_Kap6_PromptBaseline",
    "tags":     ["M03", "prompt-engineering", "research"]
}

In [18]:
# 2. Nur den Referenzlauf in einem expliziten Tracing-Kontext ausführen
from langchain_core.tracers.context import tracing_v2_enabled

traced_chain = (prompt | llm | parser).with_config(**run_cfg)

with tracing_v2_enabled(
    project_name="M03-Prompt-Engineering",
    tags=run_cfg["tags"],
):
    traced_response = traced_chain.invoke({
        "thema": "Nicht-im-Korpus-Antworten",
        "zielgruppe": "Briefing-Agent-Entwicklung"
    })

mprint("### Trace erstellt")
mprint('---')
mprint(traced_response)

### Trace erstellt

---

Nicht-im-Korpus-Antworten sind Antworten auf Fragen, zu denen die bereitgestellten Projektunterlagen oder wissenschaftlichen Dokumente keine ausreichenden Belege enthalten.  
Ein Briefing-Agent sollte solche Fälle erkennen, die Wissensgrenze transparent benennen und weder spekulieren noch unbelegte Inhalte als Fakten darstellen.  
Stattdessen sollte er um Präzisierung bitten, eine gezielte externe Recherche vorschlagen oder klar gekennzeichnetes Allgemeinwissen getrennt von korpusbasierten Aussagen ausgeben.

In [19]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M03-Prompt-Engineering", limit=3, show_steps=True)

## LangSmith Trace — `M03-Prompt-Engineering`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `M03_Kap6_PromptBaseline` | ✅ success | 2.7s | 0 |


### Steps — letzter Run: `M03_Kap6_PromptBaseline`

| # | Typ | Name | Status | Dauer |
|---|-----|------|--------|-------|
| 1 | `parser` | `StrOutputParser` | ✅ | 0.0s |
| 2 | `llm` | `ChatOpenAI` | ✅ | 2.7s |
| 3 | `prompt` | `ChatPromptTemplate` | ✅ | 0.0s |

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**
- Einen klaren System-Prompt für einen einfachen Meeting-Briefing-Agent schreiben.
- Mit 1-2 Beispielanfragen testen.

**✅ Erledigt wenn:** `mein_system_prompt` enthält mindestens 20 Zeichen und der Agent gibt für eine Testfrage eine sinnvolle Antwort zurück.


In [20]:
# Grundlagen: System-Prompt + Agent
from langchain_core.tools import tool

@tool
def suche_research_signal(thema: str) -> str:
    """Gibt Demo-Informationen zu einem Research-Thema zurück."""
    pass

@tool
def pruefe_korpus(thema: str) -> str:
    """Prüft eine grobe Demo-Korpusabdeckung."""
    pass

**Aufbau**
1. Einen Agenten mit 2-3 Briefing-Tools erstellen, zum Beispiel Korpuscheck, Signalextraktion und Quellenhinweis.
2. Zwei unterschiedliche System-Prompts schreiben: einmal knapp, einmal vorsichtig mit Unsicherheitsmarkierung.
3. 3 Testfragen definieren und beide Varianten vergleichen.
4. Kurz dokumentieren, welche Prompt-Version für den Meeting-Briefing-Agent besser passt und warum.

**✅ Erledigt wenn:** `meine_varianten` enthält mindestens zwei Einträge und der Unterschied beider Prompts ist in einer Zeile dokumentiert.


In [21]:
# Aufbau: Zwei Prompt-Varianten vergleichen
meine_varianten = {
    "knapp": "...",
    "vorsichtig": "...",
}

pass

**Vertiefung**
- Ein Few-Shot-Beispiel in einer der beiden Prompt-Varianten ergänzen.
- Eine klare Regel formulieren, wann der Agent Tools vermeiden soll.

**✅ Erledigt wenn:** `mein_few_shot_prompt` und `tool_vermeidungsregel` sind gesetzt; LangSmith-Tracing ist aktiv.


In [22]:
# Vertiefung: Few-Shot + Tool-Vermeidungsregel
mein_few_shot_prompt = """Beispiel 1:
Frage: Was bedeutet RAG?
Antwort: definition

Beispiel 2:
Frage: Wie unterscheidet sich RAG von Fine-Tuning?
Antwort: vergleich

# ...

_IncompleteInputError: incomplete input (4140899669.py, line 2)

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)
- [Modellauswahl](https://editor.p5js.org/ralf.bendig.rb/full/8BbTi8Ico)
- [Modellsteuerung](https://editor.p5js.org/ralf.bendig.rb/full/um423ggnD)
- [LLM-Parameter](https://editor.p5js.org/ralf.bendig.rb/full/LBc3t3yP4)



# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Prompt Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/prompt-engineering.html)
- [Einsteiger-Prompts](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/einsteiger-prompts.html)
- [Tool Use & Function Calling](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/tool-use-function-calling.html)
- [Agenten-Architekturen](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/agent-architekturen.html)
- [LangChain Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langchain-best-practices.html)
- [Fine-Tuning](https://ralf-42.github.io/Agenten/03-modelle-provider-anpassung/fine-tuning.html)

